# BioJEPA v0.6 Data Prep - Notebook 1: Genes and Splits

This notebook handles:
1. Loading GEARS k562e for official train/val/test perturbation splits
2. Defining cumulative split strategy for all datasets
3. Collecting gene universe from all 6 datasets
4. Selecting final 16,384 genes (k562e-first priority)
5. Generating per-dataset gene masks

**Outputs:**
- `gene_names.json` - ordered list of gene symbols
- `gene_to_idx.json` - ENSG -> index mapping
- `holdout_perturbations.json` - test split perturbations
- `dataset_gene_masks.json` - per-dataset boolean masks
- `cell_type_to_id.json` - cell type name to ID mapping
- `dataset_splits.json` - train/val/test perturbation sets per dataset

In [1]:
from pathlib import Path
from gears import PertData
from collections import defaultdict
from scipy.sparse import issparse
import pandas as pd
import numpy as np
import scanpy as sc
import json
import gc

In [2]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')

datasets = {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e': ref_dir / 'rep1e' / 'rpe1_raw_singlecell_01.h5ad',
    'k562gw': ref_dir / 'k562gw' / 'K562_gwps_raw_singlecell_01.h5ad',
    'adamson': ref_dir / 'adamson' / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad',
    'norman': ref_dir / 'norman' / 'NormanWeissman2019_filtered.h5ad',
    'sciplex': ref_dir / 'sciplex' / 'SrivatsanTrapnell2020_sciplex3.h5ad',
}

N_GENES = 10000
MIN_CELLS = 10
CHUNK_SIZE = 5000
VAL_PCT = 0.05
TEST_PCT = 0.10
SEED = 42
np.random.seed(SEED)

## Step 1: Load GEARS k562e Splits

GEARS provides official train/val/test splits for the k562 essential dataset. These are authoritative and will form the basis for all other dataset splits.

In [3]:
def clean_gears_name(name):
    if name.endswith('+ctrl'):
        return name.replace('+ctrl', '')
    if name.startswith('ctrl+'):
        return name.replace('ctrl+', '')
    if name == 'ctrl':
        return 'control'
    return name.strip()

In [4]:
pert_data = PertData(ref_dir / 'k562e')
pert_data.load(data_name='replogle_k562_essential')
pert_data.prepare_split(split='simulation', seed=1)

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['C7orf26+ctrl' 'C14orf178+ctrl' 'RPS10-NUDT3+ctrl' 'SEM1+ctrl' 'FAU+ctrl']
Local copy of pyg dataset is detected. Loading...
Done!
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:0
combo_seen1:0
combo_seen2:0
unseen_single:272
Done!


here1


In [5]:
k562e_train_perts = {clean_gears_name(p) for p in pert_data.set2conditions['train']}
k562e_val_perts = {clean_gears_name(p) for p in pert_data.set2conditions['val']}
k562e_test_perts = {clean_gears_name(p) for p in pert_data.set2conditions['test']}

print(f'k562e splits - train: {len(k562e_train_perts)} | val: {len(k562e_val_perts)} | test: {len(k562e_test_perts)}')

k562e splits - train: 734 | val: 82 | test: 272


In [6]:
cumulative_train = set(k562e_train_perts)
cumulative_val = set(k562e_val_perts)
cumulative_test = set(k562e_test_perts)

## Step 2: Cumulative Split Strategy

For each additional dataset, we:
1. Check if perturbations already exist in k562e splits (reuse those assignments)
2. For new perturbations, add holdouts only if current percentage is below target

In [7]:
def get_dataset_splits(ds_perts, cumulative_train, cumulative_val, cumulative_test, ds_name):
    ds_perts = set(ds_perts)
    n_total = len(ds_perts)
    if n_total == 0:
        return set(), set(), set()

    already_test = ds_perts & cumulative_test
    already_val = ds_perts & cumulative_val
    already_train = ds_perts & cumulative_train
    unassigned = list(ds_perts - cumulative_train - cumulative_val - cumulative_test)
    np.random.shuffle(unassigned)

    current_test_pct = len(already_test) / n_total
    current_val_pct = len(already_val) / n_total

    target_test = int(n_total * TEST_PCT)
    target_val = int(n_total * VAL_PCT)

    needed_test = max(0, target_test - len(already_test))
    needed_val = max(0, target_val - len(already_val))

    new_test = set(unassigned[:needed_test])
    new_val = set(unassigned[needed_test:needed_test + needed_val])
    new_train = set(unassigned[needed_test + needed_val:])

    cumulative_test.update(new_test)
    cumulative_val.update(new_val)
    cumulative_train.update(new_train)

    ds_train = already_train | new_train
    ds_val = already_val | new_val
    ds_test = already_test | new_test

    print(f'{ds_name}: Total {n_total} | Already test: {len(already_test)} ({current_test_pct:.1%}) | '
          f'Added test: {len(new_test)} | Final test: {len(ds_test)} ({len(ds_test)/n_total:.1%})')

    return ds_train, ds_val, ds_test

## Step 3: Collect Gene Universe

For k562e_raw: take ALL genes (no filter)
For other datasets: filter to genes with ncells >= 10

In [8]:
def count_cells_per_gene_chunked(adata, chunk_size=50000):
    n_cells = adata.n_obs
    n_genes = adata.n_vars
    gene_cell_counts = np.zeros(n_genes, dtype=np.int64)

    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)
        chunk = adata.X[start:end]
        if issparse(chunk):
            chunk = chunk.toarray()
        gene_cell_counts += (chunk > 0).sum(axis=0).astype(np.int64)
        del chunk
        gc.collect()
        if start % (chunk_size * 10) == 0 and start > 0:
            print(f'  Processed {start:,} / {n_cells:,} cells')

    return gene_cell_counts

In [9]:
def is_valid(val):
    if pd.isna(val):
        return False
    if isinstance(val, str) and (val.strip() == '' or val.strip().lower() == 'nan'):
        return False
    return True

In [10]:
def print_list_head(print_list, n=10):
    print({j:print_list[j] for j in list(print_list.keys())[:n]})

### k562e_raw (primary - take all genes)

In [11]:
ds_adata = sc.read_h5ad(datasets['k562e_raw'], backed='r')
var_df = ds_adata.var
k562e_genes = {idx: row['gene_name'] for idx, row in var_df.iterrows() if is_valid(idx)}
print(f'k562e_raw: {len(k562e_genes)} genes (all, no filter)')

k562e_raw: 8563 genes (all, no filter)


In [12]:
dataset_genes = {'k562e_raw': set(k562e_genes.keys())}
all_genes = dict(k562e_genes)
gene_cell_counts = {}

print_list_head(all_genes)

{'ENSG00000237491': 'LINC01409', 'ENSG00000228794': 'LINC01128', 'ENSG00000188976': 'NOC2L', 'ENSG00000187961': 'KLHL17', 'ENSG00000188290': 'HES4', 'ENSG00000187608': 'ISG15', 'ENSG00000078808': 'SDF4', 'ENSG00000176022': 'B3GALT6', 'ENSG00000160087': 'UBE2J2', 'ENSG00000131584': 'ACAP3'}


### rep1e (filter ncells >= 10)

In [13]:
ds = 'rep1e'
ds_adata = sc.read_h5ad(datasets[ds], backed='r')
var_df = ds_adata.var.copy()

In [14]:
if 'ncells' in var_df.columns:
    print(f'{ds} using ncells')
    gene_counts = var_df['ncells'].values
else:
    print(f'Computing cell counts for {ds}')
    gene_counts = count_cells_per_gene_chunked(ds_adata, CHUNK_SIZE)

Computing cell counts for rep1e
  Processed 50,000 / 247,914 cells
  Processed 100,000 / 247,914 cells
  Processed 150,000 / 247,914 cells
  Processed 200,000 / 247,914 cells


In [15]:
valid_mask = gene_counts >= MIN_CELLS
var_df = var_df[valid_mask]
gene_counts_filtered = gene_counts[valid_mask]
var_df['cell_count'] = gene_counts_filtered
var_df.head()

,gene_name,chr,start,end,class,strand,length,in_matrix,mean,std,cv,fano,cell_count
gene_id,,,,,,,,,,,,,
ENSG00000188976,NOC2L,chr1,944203,959309,gene_version11,-,15106,True,0.997140,1.149519,1.152816,1.325185,145522
ENSG00000187583,PLEKHN1,chr1,966482,975865,gene_version11,+,9383,True,0.131328,0.377933,2.877783,1.087609,29386
ENSG00000188290,HES4,chr1,998962,1000172,gene_version10,-,1210,True,0.732129,1.526109,2.084482,3.181148,104265
ENSG00000187608,ISG15,chr1,1001138,1014540,gene_version10,+,13402,True,0.455956,1.260789,2.765152,3.486272,80037
ENSG00000188157,AGRN,chr1,1020120,1056118,gene_version15,+,35998,True,0.346108,0.648378,1.873341,1.214633,67400


In [16]:
rep1e_genes = {idx: row['gene_name'] for idx, row in var_df.iterrows() if is_valid(idx)}
dataset_genes[ds] = set(rep1e_genes.keys())

In [17]:
for idx, row in var_df.iterrows():
    if not is_valid(idx):
        continue
    if idx not in all_genes:
        all_genes[idx] = row['gene_name']
    gene_cell_counts[idx] = gene_cell_counts.get(idx, 0) + row['cell_count']

print(f'{ds}: {len(rep1e_genes)} genes (filtered ncells >= {MIN_CELLS})')

rep1e: 8749 genes (filtered ncells >= 10)


In [18]:
del ds_adata
gc.collect()

0

### k562gw (filter ncells >= 10)

In [19]:
ds = 'k562gw'
ds_adata = sc.read_h5ad(datasets[ds], backed='r')
var_df = ds_adata.var.copy()

In [20]:
if 'ncells' in var_df.columns:
    print(f'{ds} using ncells')
    gene_counts = var_df['ncells'].values
else:
    print(f'Computing cell counts for {ds}')
    gene_counts = count_cells_per_gene_chunked(ds_adata, CHUNK_SIZE)

Computing cell counts for k562gw
  Processed 50,000 / 1,989,578 cells
  Processed 100,000 / 1,989,578 cells
  Processed 150,000 / 1,989,578 cells
  Processed 200,000 / 1,989,578 cells
  Processed 250,000 / 1,989,578 cells
  Processed 300,000 / 1,989,578 cells
  Processed 350,000 / 1,989,578 cells
  Processed 400,000 / 1,989,578 cells
  Processed 450,000 / 1,989,578 cells
  Processed 500,000 / 1,989,578 cells
  Processed 550,000 / 1,989,578 cells
  Processed 600,000 / 1,989,578 cells
  Processed 650,000 / 1,989,578 cells
  Processed 700,000 / 1,989,578 cells
  Processed 750,000 / 1,989,578 cells
  Processed 800,000 / 1,989,578 cells
  Processed 850,000 / 1,989,578 cells
  Processed 900,000 / 1,989,578 cells
  Processed 950,000 / 1,989,578 cells
  Processed 1,000,000 / 1,989,578 cells
  Processed 1,050,000 / 1,989,578 cells
  Processed 1,100,000 / 1,989,578 cells
  Processed 1,150,000 / 1,989,578 cells
  Processed 1,200,000 / 1,989,578 cells
  Processed 1,250,000 / 1,989,578 cells
  Proc

In [21]:
valid_mask = gene_counts >= MIN_CELLS
var_df = var_df[valid_mask]
gene_counts_filtered = gene_counts[valid_mask]
var_df['cell_count'] = gene_counts_filtered
var_df.head()

,gene_name,chr,start,end,class,strand,length,in_matrix,mean,std,cv,fano,cell_count
gene_id,,,,,,,,,,,,,
ENSG00000237491,LINC01409,chr1,778747,810065,gene_version10,+,31318,True,0.116626,0.349971,3.000803,1.050194,214242
ENSG00000228794,LINC01128,chr1,825138,868202,gene_version9,+,43064,True,0.182850,0.437274,2.391434,1.045713,325879
ENSG00000188976,NOC2L,chr1,944203,959309,gene_version11,-,15106,True,1.415674,1.397208,0.986957,1.378984,1391744
ENSG00000187961,KLHL17,chr1,960584,965719,gene_version14,+,5135,True,0.105599,0.330678,3.131439,1.035497,196093
ENSG00000188290,HES4,chr1,998962,1000172,gene_version10,-,1210,True,0.242700,0.550596,2.268630,1.249098,391564


In [22]:
k562gw_genes = {idx: row['gene_name'] for idx, row in var_df.iterrows() if is_valid(idx)}
dataset_genes[ds] = set(k562gw_genes.keys())

In [23]:
for idx, row in var_df.iterrows():
    if not is_valid(idx):
        continue
    if idx not in all_genes:
        all_genes[idx] = row['gene_name']
    gene_cell_counts[idx] = gene_cell_counts.get(idx, 0) + row['cell_count']

print(f'{ds}: {len(k562gw_genes)} genes (filtered ncells >= {MIN_CELLS})')

k562gw: 8248 genes (filtered ncells >= 10)


In [24]:
del ds_adata
gc.collect()

0

### adamson (filter ncells >= 10)

In [25]:
ds = 'adamson'
ds_adata = sc.read_h5ad(datasets[ds], backed='r')
var_df = ds_adata.var.copy()

In [26]:
if 'ncells' in var_df.columns:
    print(f'{ds} using ncells')
    gene_counts = var_df['ncells'].values
else:
    print(f'Computing cell counts for {ds}')
    gene_counts = count_cells_per_gene_chunked(ds_adata, CHUNK_SIZE)

adamson using ncells


In [27]:
valid_mask = gene_counts >= MIN_CELLS
var_df = var_df[valid_mask]
gene_counts_filtered = gene_counts[valid_mask]
var_df['cell_count'] = gene_counts_filtered
var_df.head()

,ensembl_id,ncounts,ncells,cell_count
gene_symbol,,,,
MIR1302-10,ENSG00000243485,11.0,11,11
RP11-34P13.8,ENSG00000239945,43.0,43,43
AL627309.1,ENSG00000237683,337.0,333,333
AP006222.2,ENSG00000228463,48.0,48,48
RP11-206L10.3,ENSG00000235373,20.0,20,20


In [28]:
adamson_genes = {row['ensembl_id']: row.name for _, row in var_df.iterrows() if is_valid(row.get('ensembl_id'))}
dataset_genes[ds] = set(adamson_genes.keys())

In [29]:
for _, row in var_df.iterrows():
    idx = row.get('ensembl_id')
    if not is_valid(idx):
        continue
    if idx not in all_genes:
        all_genes[idx] = row.name
    gene_cell_counts[idx] = gene_cell_counts.get(idx, 0) + row['cell_count']

print(f'{ds}: {len(adamson_genes)} genes (filtered ncells >= {MIN_CELLS})')

adamson: 18276 genes (filtered ncells >= 10)


In [30]:
del ds_adata
gc.collect()

0

### norman (filter ncells >= 10)

In [31]:
ds = 'norman'
ds_adata = sc.read_h5ad(datasets[ds], backed='r')
var_df = ds_adata.var.copy()

In [32]:
if 'ncells' in var_df.columns:
    print(f'{ds} using ncells')
    gene_counts = var_df['ncells'].values
else:
    print(f'Computing cell counts for {ds}')
    gene_counts = count_cells_per_gene_chunked(ds_adata, CHUNK_SIZE)

norman using ncells


In [33]:
valid_mask = gene_counts >= MIN_CELLS
var_df = var_df[valid_mask]
gene_counts_filtered = gene_counts[valid_mask]
var_df['cell_count'] = gene_counts_filtered
var_df.head()

,ensemble_id,ncounts,ncells,cell_count
RP11-34P13.3,ENSG00000243485,29.0,29,29
RP11-34P13.7,ENSG00000238009,266.0,265,265
RP11-34P13.8,ENSG00000239945,10.0,10,10
FO538757.3,ENSG00000279928,12.0,12,12
FO538757.2,ENSG00000279457,77489.0,52291,52291


In [34]:
norman_genes = {row['ensemble_id']: row.name for _, row in var_df.iterrows() if is_valid(row.get('ensemble_id'))}
dataset_genes[ds] = set(norman_genes.keys())

In [35]:
for _, row in var_df.iterrows():
    idx = row.get('ensemble_id')
    if not is_valid(idx):
        continue
    if idx not in all_genes:
        all_genes[idx] = row.name
    gene_cell_counts[idx] = gene_cell_counts.get(idx, 0) + row['cell_count']

print(f'{ds}: {len(norman_genes)} genes (filtered ncells >= {MIN_CELLS})')

norman: 20265 genes (filtered ncells >= 10)


In [36]:
del ds_adata
gc.collect()

0

### sciplex (filter ncells >= 10 if available)

In [37]:
ds = 'sciplex'
ds_adata = sc.read_h5ad(datasets[ds], backed='r')
var_df = ds_adata.var.copy()

In [38]:
if 'ncells' in var_df.columns:
    print(f'{ds} using ncells')
    gene_counts = var_df['ncells'].values
else:
    print(f'Computing cell counts for {ds}')
    gene_counts = count_cells_per_gene_chunked(ds_adata, CHUNK_SIZE)

Computing cell counts for sciplex
  Processed 50,000 / 799,317 cells
  Processed 100,000 / 799,317 cells
  Processed 150,000 / 799,317 cells
  Processed 200,000 / 799,317 cells
  Processed 250,000 / 799,317 cells
  Processed 300,000 / 799,317 cells
  Processed 350,000 / 799,317 cells
  Processed 400,000 / 799,317 cells
  Processed 450,000 / 799,317 cells
  Processed 500,000 / 799,317 cells
  Processed 550,000 / 799,317 cells
  Processed 600,000 / 799,317 cells
  Processed 650,000 / 799,317 cells
  Processed 700,000 / 799,317 cells
  Processed 750,000 / 799,317 cells


In [39]:
valid_mask = gene_counts >= MIN_CELLS
var_df = var_df[valid_mask]
gene_counts_filtered = gene_counts[valid_mask]
var_df['cell_count'] = gene_counts_filtered
var_df.head()

,ensembl_id,cell_count
gene_symbol,,
TSPAN6,ENSG00000000003,23228
TNMD,ENSG00000000005,33
DPM1,ENSG00000000419,116153
SCYL3,ENSG00000000457,41883
C1orf112,ENSG00000000460,49609


In [40]:
sciplex_genes = {row['ensembl_id']: row.name for _, row in var_df.iterrows() if is_valid(row.get('ensembl_id'))}
dataset_genes[ds] = set(sciplex_genes.keys())

In [41]:
for _, row in var_df.iterrows():
    idx = row.get('ensembl_id')
    if not is_valid(idx):
        continue
    if idx not in all_genes:
        all_genes[idx] = row.name
    gene_cell_counts[idx] = gene_cell_counts.get(idx, 0) + row['cell_count']

print(f'{ds}: {len(sciplex_genes)} genes (filtered ncells >= {MIN_CELLS})')

sciplex: 53017 genes (filtered ncells >= 10)


In [42]:
del ds_adata
gc.collect()

0

In [43]:
print(f'\nTotal unique genes across all datasets: {len(all_genes)}')
for ds, genes in dataset_genes.items():
    print(f'  {ds}: {len(genes)}')


Total unique genes across all datasets: 54064
  k562e_raw: 8563
  rep1e: 8749
  k562gw: 8248
  adamson: 18276
  norman: 20265
  sciplex: 53017


## Step 4: Select Final Genes

Strategy: k562e-first, then filtered genes from other datasets, preferring genes in multiple datasets

In [44]:
selected = list(k562e_genes.keys())
print(f'Starting with {len(selected)} genes from k562e')

Starting with 8563 genes from k562e


In [45]:
remaining_counts = {g: c for g, c in gene_cell_counts.items() if g not in k562e_genes}
print_list_head(remaining_counts)

{'ENSG00000187583': 35802, 'ENSG00000188157': 126179, 'ENSG00000224051': 89448, 'ENSG00000157911': 101571, 'ENSG00000157870': 45399, 'ENSG00000162591': 63167, 'ENSG00000158109': 95150, 'ENSG00000158286': 52467, 'ENSG00000116285': 220602, 'ENSG00000171621': 124000}


In [46]:
sorted_genes = sorted(remaining_counts.keys(), key=lambda g: -remaining_counts[g])
sorted_genes[:10]

['ENSG00000210082',
 'ENSG00000211459',
 'ENSG00000230876',
 'ENSG00000141376',
 'ENSG00000110092',
 'ENSG00000096696',
 'ENSG00000170017',
 'ENSG00000145012',
 'ENSG00000138119',
 'ENSG00000103187']

In [47]:
for g in sorted_genes:
    selected.append(g)
    if len(selected) >= N_GENES:
        break

print(f'Selected {len(selected)} genes: {len(k562e_genes)} from k562e + {len(selected) - len(k562e_genes)} from other datasets')
print(f'Top 5 added genes by cell count: {[(all_genes[g], remaining_counts[g]) for g in sorted_genes[:5]]}')

Selected 10000 genes: 8563 from k562e + 1437 from other datasets
Top 5 added genes by cell count: [('MT-RNR2', 799260), ('MT-RNR1', 794037), ('LINC00486', 646066), ('BCAS3', 578710), ('CCND1', 551213)]


In [48]:
gene_to_idx = {g: i for i, g in enumerate(selected)}
gene_names = [all_genes.get(g, g) for g in selected]

In [49]:
with open(data_dir / 'gene_names.json', 'w') as f:
    json.dump(gene_names, f)

with open(data_dir / 'gene_to_idx.json', 'w') as f:
    json.dump(gene_to_idx, f)

## Step 5: Generate Per-Dataset Gene Masks

In [50]:
dataset_masks = {}
for ds_name, ds_genes in dataset_genes.items():
    mask = np.zeros(len(selected), dtype=bool)
    for g in ds_genes:
        if g in gene_to_idx:
            mask[gene_to_idx[g]] = True
    dataset_masks[ds_name] = mask.tolist()
    print(f'{ds_name}: {mask.sum()} / {len(selected)} genes covered')

k562e_raw: 8563 / 10000 genes covered
rep1e: 8162 / 10000 genes covered
k562gw: 8248 / 10000 genes covered
adamson: 9446 / 10000 genes covered
norman: 9630 / 10000 genes covered
sciplex: 9955 / 10000 genes covered


In [51]:
with open(data_dir / 'dataset_gene_masks.json', 'w') as f:
    json.dump(dataset_masks, f)

## Step 6: Extract Dataset Perturbations and Assign Splits

In [52]:
dataset_splits = {
    'k562e_raw': {
        'train': list(k562e_train_perts),
        'val': list(k562e_val_perts),
        'test': list(k562e_test_perts)
    }
}

In [53]:
def extract_perturbations(ds_key, condition_col='condition'):
    ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
    
    pert_col = None
    for col in [condition_col, 'perturbation', 'gene']:
        if col in ds_adata.obs.columns:
            pert_col = col
            break
    if pert_col is None:
        print(f'Warning: No perturbation column found in {ds_key}')
        return set()
    
    perts = ds_adata.obs[pert_col].unique().tolist()
    ctrl_keywords = ['control', 'ctrl', 'non-targeting', 'vehicle', 'dmso']
    cleaned = []
    for p in perts:
        if is_valid(p):
            cp = clean_gears_name(str(p))
            if cp != 'control' and str(p).lower() not in ctrl_keywords:
                cleaned.append(cp)
    return set(cleaned)

In [54]:
for ds_key in ['rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']:
    print(f'\nProcessing {ds_key}...')
    perts = extract_perturbations(ds_key)
    print(f'  Found {len(perts)} perturbations')
    
    train, val, test = get_dataset_splits(perts, cumulative_train, cumulative_val, cumulative_test, ds_key)
    
    dataset_splits[ds_key] = {
        'train': list(train),
        'val': list(val),
        'test': list(test)
    }


Processing rep1e...
  Found 2393 perturbations
rep1e: Total 2393 | Already test: 272 (11.4%) | Added test: 0 | Final test: 272 (11.4%)

Processing k562gw...
  Found 9866 perturbations
k562gw: Total 9866 | Already test: 272 (2.8%) | Added test: 714 | Final test: 986 (10.0%)

Processing adamson...
  Found 114 perturbations
adamson: Total 114 | Already test: 0 (0.0%) | Added test: 11 | Final test: 11 (9.6%)

Processing norman...
  Found 236 perturbations
norman: Total 236 | Already test: 15 (6.4%) | Added test: 8 | Final test: 23 (9.7%)

Processing sciplex...
  Found 188 perturbations
sciplex: Total 188 | Already test: 0 (0.0%) | Added test: 18 | Final test: 18 (9.6%)


## Step 7: Save Outputs

In [55]:
cell_type_to_id = {
    'K562': 0,
    'RPE1': 1,
    'A549': 2,
    'MCF7': 3,
    'unknown': 4
}

In [56]:

with open(data_dir / 'holdout_perturbations.json', 'w') as f:
    json.dump(list(cumulative_test), f)

with open(data_dir / 'cell_type_to_id.json', 'w') as f:
    json.dump(cell_type_to_id, f)

with open(data_dir / 'dataset_splits.json', 'w') as f:
    json.dump(dataset_splits, f)

print('Saved outputs to', data_dir)

Saved outputs to /Users/djemec/data/jepa/v0_6


## Summary

In [57]:
print('=== Data Prep Notebook 1 Complete ===')
print(f'Selected genes: {len(selected)}')
print(f'Total holdout perturbations: {len(cumulative_test)}')
print(f'Total val perturbations: {len(cumulative_val)}')
print(f'Total train perturbations: {len(cumulative_train)}')
print('\nPer-dataset splits:')
for ds, splits in dataset_splits.items():
    print(f"  {ds}: train={len(splits['train'])} val={len(splits['val'])} test={len(splits['test'])}")

=== Data Prep Notebook 1 Complete ===
Selected genes: 10000
Total holdout perturbations: 1023
Total val perturbations: 515
Total train perturbations: 8786

Per-dataset splits:
  k562e_raw: train=734 val=82 test=272
  rep1e: train=2002 val=119 test=272
  k562gw: train=8387 val=493 test=986
  adamson: train=98 val=5 test=11
  norman: train=202 val=11 test=23
  sciplex: train=161 val=9 test=18
